In [ ]:
import os
import sys
import copy
import time
import math
import hashlib
import random
import itertools
from collections import deque

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import OneHotEncoder
from torch.distributions import Categorical


In [ ]:
SEED = 97620260313
N_RUNS = 5
MAX_STEPS = 30
UNLEARN_BATCH = 64
KS = [1, 5, 10]
PATIENCE = max(10, int(0.1 * 10000 / 100))

LOG_DIR = 'Run_Best_Logs'
MODELS_DIR = os.path.join(LOG_DIR, 'models')
METRICS_CSV = os.path.join(LOG_DIR, 'run_best_metrics.csv')
os.makedirs(MODELS_DIR, exist_ok=True)

from pathlib import Path
SCRIPT_ROOT = Path("E:/Kuliah/Kuliah/Kuliah/PRODI/Semester 7/ProSkripCode")
DATA_DIR = 'C:/Bob/ml-1m' 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

HARDCODED_BEST_CONFIGS = {
    "Normal_1_percent": {
        "source_csv": str((SCRIPT_ROOT / "Raw" / "Normal" / "1_percent" / "tuning_full_results.csv").resolve()),
        "split_mode": "normal",
        "methods": {
            "Ye_multi": {
                "train_lr": 0.001,
                "gamma": 0.97,
                "hidden_dim": 256,
                "train_batch": 2,
                "unlearn_lr": 0.001,
                "unlearn_iters": 2000,
                "lambda_retain": 1.0,
                "retain_drop_pp": 1.5050167224080202,
                "forget_drop_pp": 30.000000000000014,
                "reason": "Highest forget drop among feasible Ye_multi candidates while staying within the 2 pp retain-drop limit.",
            },
            "New_True_inf": {
                "train_lr": 0.001,
                "gamma": 0.98,
                "hidden_dim": 256,
                "train_batch": 2,
                "unlearn_lr": 0.001,
                "unlearn_iters": 1500,
                "lambda_retain": 0.8,
                "retain_drop_pp": 1.705685618729097,
                "forget_drop_pp": 38.333333333333336,
                "reason": "Best feasible New_True_inf configuration by forget drop with retain drop still below the 2 pp cap.",
            },
            "Gradient_Ascent": {
                "train_lr": 0.001,
                "gamma": 0.97,
                "hidden_dim": 256,
                "train_batch": 2,
                "unlearn_lr": 0.001,
                "unlearn_iters": 1000,
                "lambda_retain": 0.0,
                "retain_drop_pp": 0.3010033444815985,
                "forget_drop_pp": 3.3333333333333437,
                "reason": "Best feasible Gradient_Ascent row under the retain-drop constraint; the forget gain is much smaller than the other methods.",
            },
        },
    },
    "Demography_1_percent": {
        "source_csv": str((SCRIPT_ROOT / "Raw" / "Demography" / "1_percent" / "tuning_full_results.csv").resolve()),
        "split_mode": "demography",
        "methods": {
            "Ye_multi": {
                "train_lr": 0.001,
                "gamma": 0.99,
                "hidden_dim": 256,
                "train_batch": 4,
                "unlearn_lr": 0.001,
                "unlearn_iters": 1500,
                "lambda_retain": 1.0,
                "retain_drop_pp": 1.53820431365993,
                "forget_drop_pp": 25.423728813559325,
                "reason": "Best feasible Ye_multi configuration for the demography split; highest forget drop among rows within the 2 pp retain-drop bound.",
            },
            "New_True_inf": {
                "train_lr": 0.001,
                "gamma": 0.99,
                "hidden_dim": 256,
                "train_batch": 4,
                "unlearn_lr": 0.001,
                "unlearn_iters": 1500,
                "lambda_retain": 0.7,
                "retain_drop_pp": 1.8391573315499121,
                "forget_drop_pp": 49.152542372881356,
                "reason": "Best demography configuration overall for New_True_inf by forget drop under the 2 pp retain-drop cap.",
            },
            "Gradient_Ascent": {
                "train_lr": 0.001,
                "gamma": 0.98,
                "hidden_dim": 256,
                "train_batch": 2,
                "unlearn_lr": 0.001,
                "unlearn_iters": 500,
                "lambda_retain": 0.0,
                "retain_drop_pp": 1.9060357799699112,
                "forget_drop_pp": 6.779661016949157,
                "reason": "Best feasible Gradient_Ascent row under the 2 pp retain-drop constraint; it has the largest forget drop among the feasible demography rows.",
            },
        },
    },
    "UGP_Analysis": {
        "source_csv": str((SCRIPT_ROOT / "Raw" / "results_ugp_analysis" / "metrics" / "relearn_metrics.csv").resolve()),
        "split_mode": "ugp_setting",
        "methods": {
            "Ye_multi": {
                "train_lr": 0.001,
                "gamma": 0.97,
                "hidden_dim": 256,
                "train_batch": 2,
                "unlearn_lr": 0.001,
                "unlearn_iters": 1500,
                "lambda_retain": 1.0,
                "setting_id": 25,
                "setting_type": "occupation",
                "setting_value_raw": 15,
                "setting_label": "occupation_15_scientist",
                "retain_drop_pp": 0.8862876254180585,
                "forget_drop_pp": 38.33333333333334,
                "reason": "Best UGP scientist split for Ye_multi by forget drop under the 2 pp retain-drop limit.",
            },
            "New_True_inf": {
                "train_lr": 0.001,
                "gamma": 0.98,
                "hidden_dim": 256,
                "train_batch": 2,
                "unlearn_lr": 0.001,
                "unlearn_iters": 1500,
                "lambda_retain": 0.8,
                "setting_id": 28,
                "setting_type": "occupation",
                "setting_value_raw": 19,
                "setting_label": "occupation_19_unemployed",
                "retain_drop_pp": 1.7892976588628762,
                "forget_drop_pp": 40.0,
                "reason": "Best UGP unemployed split for New_True_inf with the largest forget drop among feasible rows.",
            },
            "Gradient_Ascent": {
                "train_lr": 0.001,
                "gamma": 0.99,
                "hidden_dim": 256,
                "train_batch": 4,
                "unlearn_lr": 0.0001,
                "unlearn_iters": 2000,
                "lambda_retain": 0.0,
                "setting_id": 7,
                "setting_type": "age",
                "setting_value_raw": 45,
                "setting_label": "age_45_49",
                "retain_drop_pp": -0.050167224080271966,
                "forget_drop_pp": 4.999999999999999,
                "reason": "Best UGP age 45-49 split for Gradient_Ascent under the retain-drop constraint.",
            },
        },
    },
}

In [ ]:
def make_seed(*args):
    key = f'{SEED}|{''|''.join(str(a) for a in args)}'.encode('utf-8')
    return int(hashlib.sha256(key).hexdigest()[:8], 16)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed % (2**31 - 1))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed % (2**31 - 1))

set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
torch.use_deterministic_algorithms(True, warn_only=True)

class PolicyNet(nn.Module):
    def __init__(self, state_dim, num_actions, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.logits = nn.Linear(hidden_dim, num_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.logits(x)

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, *args):
        self.buffer.append(args)
    def sample(self, batch_size, device):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, done = map(lambda x: torch.tensor(np.array(x)), zip(*batch))
        return s.float().to(device), a.long().to(device), r.float().to(device), s_next.float().to(device), done.float().to(device)
    def __len__(self): return len(self.buffer)


In [ ]:
def train_policy_gradient_batched(env, policy_net, optimizer, num_episodes=10000, gamma=0.99, max_steps_per_ep=30, batch_size=1, log_interval=100):
    returns_log = []
    best_avg_return = -float('inf')
    patience_counter = 0
    ep = 0

    while ep < num_episodes:
        batch_log_probs, batch_returns, batch_ep_rets = [], [], []

        for _ in range(batch_size):
            if ep >= num_episodes:
                break
            state = env.reset()
            lps, rews = [], []

            for _ in range(max_steps_per_ep):
                st = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                probs = F.softmax(policy_net(st), dim=-1).squeeze(0)
                m = Categorical(probs)
                a = m.sample()
                next_state, rew, done = env.step(a.item())
                lps.append(m.log_prob(a))
                rews.append(rew)
                if done:
                    break
                state = next_state

            G, rets = 0.0, []
            for r in reversed(rews):
                G = r + gamma * G
                rets.insert(0, G)
            rets = torch.tensor(rets, dtype=torch.float32).to(DEVICE)
            if len(rets) > 1:
                rets = (rets - rets.mean()) / (rets.std() + 1e-10)

            batch_log_probs.append(torch.stack(lps))
            batch_returns.append(rets)
            batch_ep_rets.append(float(sum(rews)))
            ep += 1

        loss = sum(-(lp * r).sum() for lp, r in zip(batch_log_probs, batch_returns))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        returns_log.extend(batch_ep_rets)

        if ep % log_interval == 0 and ep > 0:
            avg_ret = np.mean(returns_log[-log_interval:])
            print(f'  ep {ep} | avg_ret={avg_ret:.3f} | best={best_avg_return:.3f}')
            if avg_ret > best_avg_return + 0.01:
                best_avg_return = avg_ret
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  Early stop at ep {ep}')
                break
    return returns_log

def evaluate_policy(policy_net, trajectories, build_state_fn, candidate_movies, K=10):
    hits, ndcgs = [], []
    candidate_movies = np.array(candidate_movies)

    for traj in trajectories:
        uid = traj['user_id']
        movies = traj['movies']
        if len(movies) < 5:
            continue

        split = len(movies) // 2
        future = set(movies[split:])
        state = build_state_fn(uid, movies[:split])
        state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)

        with torch.no_grad():
            probs = F.softmax(policy_net(state_t), dim=-1).squeeze(0).cpu().numpy()

        topk_movies = candidate_movies[np.argsort(-probs, kind='stable')[:K]]
        hits.append(int(any(m in future for m in topk_movies)))

        dcg = sum(1.0 / np.log2(rank + 2) for rank, m in enumerate(topk_movies) if m in future)
        idcg = sum(1.0 / np.log2(rank + 2) for rank in range(min(len(future), K)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return (float(np.mean(hits)) if hits else 0.0, float(np.mean(ndcgs)) if ndcgs else 0.0)

def eval_all_ks(net, ret_trajs, for_trajs, all_trajs):
    out = {}
    for K in KS:
        h_r, n_r = evaluate_policy(net, ret_trajs, build_state_fn, candidate_movies, K=K)
        h_f, n_f = evaluate_policy(net, for_trajs, build_state_fn, candidate_movies, K=K)
        h_c, n_c = evaluate_policy(net, all_trajs, build_state_fn, candidate_movies, K=K)
        out[K] = (h_r, n_r, h_f, n_f, h_c, n_c)
    return out


In [ ]:
class MovieLensEnv:
    def __init__(self, trajectories, build_state_fn, candidate_movies):
        self.trajs = trajectories
        self.build_state_fn = build_state_fn
        self.candidate_movies = np.array(candidate_movies)
        self.num_actions = len(candidate_movies)

    def reset(self):
        traj = self.trajs[np.random.randint(len(self.trajs))]
        self.user_id = traj['user_id']
        self.movies = traj['movies']
        self.ratings = traj['ratings']
        self.t = 1
        self.future_dict = {m: r for m, r in zip(self.movies[self.t:], self.ratings[self.t:])}
        return self.build_state_fn(self.user_id, self.movies[: self.t])

    @staticmethod
    def _rating_to_reward(rating):
        if rating >= 5: return 1.0
        if rating >= 4: return 0.5
        return 0.0

    def step(self, action_idx):
        rec_movie = self.candidate_movies[action_idx]
        reward = self._rating_to_reward(self.future_dict[rec_movie]) if rec_movie in self.future_dict else 0.0
        self.t += 1
        done = self.t >= len(self.movies)
        if not done:
            self.future_dict = {m: r for m, r in zip(self.movies[self.t:], self.ratings[self.t:])}
            next_state = self.build_state_fn(self.user_id, self.movies[: self.t])
        else:
            next_state = None
        return next_state, reward, done

def collect_random_experience_forget(env_forget, num_steps, buffer):
    state = env_forget.reset()
    for _ in range(num_steps):
        action = np.random.randint(env_forget.num_actions)
        next_state, reward, done = env_forget.step(action)
        s_next = np.zeros_like(state) if next_state is None else next_state
        buffer.push(state, action, reward, s_next, float(done))
        state = env_forget.reset() if done else next_state

def collect_policy_experience(env, policy_net, num_steps, buffer):
    state = env.reset()
    for _ in range(num_steps):
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(next(policy_net.parameters()).device)
        with torch.no_grad():
            probs = F.softmax(policy_net(state_t), dim=-1).squeeze(0)
        action = Categorical(probs).sample().item()
        next_state, reward, done = env.step(action)
        s_next = np.zeros_like(state) if next_state is None else next_state
        buffer.push(state, action, reward, s_next, float(done))
        state = env.reset() if done else next_state


In [ ]:
def load_data(data_dir):
    ratings_df = pd.read_csv(os.path.join(data_dir, 'ratings.dat'), sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp'])
    movies_df = pd.read_csv(os.path.join(data_dir, 'movies.dat'), sep='::', engine='python', names=['movie_id', 'title', 'genres'], encoding='ISO-8859-1')
    users_df = pd.read_csv(os.path.join(data_dir, 'users.dat'), sep='::', engine='python', names=['user_id', 'gender', 'age', 'occupation', 'zip'])
    return ratings_df, movies_df, users_df

ratings_df, movies_df, users_df = load_data(DATA_DIR)

sample_users_list = sorted(ratings_df['user_id'].unique().tolist())
pilot_ratings = ratings_df[ratings_df['user_id'].isin(sample_users_list)].copy()
pilot_ratings.sort_values(['user_id', 'timestamp'], inplace=True)

pilot_users_df = users_df[users_df['user_id'].isin(sample_users_list)]
oh = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
user_cat_mat = oh.fit_transform(pilot_users_df[['gender', 'age', 'occupation']])
user_feat_df = pd.DataFrame(user_cat_mat, index=pilot_users_df['user_id'])

all_genres = sorted({g for s in movies_df['genres'].astype(str) for g in s.split('|')})
genre_to_idx = {g: i for i, g in enumerate(all_genres)}
num_genres = len(all_genres)

def movie_genre_vector(genres_str):
    v = np.zeros(num_genres, dtype=np.float32)
    for g in str(genres_str).split('|'):
        if g in genre_to_idx: v[genre_to_idx[g]] = 1.0
    return v

movies_df['genre_vec'] = movies_df['genres'].apply(movie_genre_vector)
movie_genre_map = {mid: movies_df.loc[movies_df['movie_id'] == mid, 'genre_vec'].values[0] for mid in movies_df['movie_id'].unique()}

state_dim = user_feat_df.shape[1] + num_genres

def build_state_fn(user_id, watched_movies):
    user_feat = user_feat_df.loc[user_id].values.astype(np.float32)
    pref_vec = np.zeros(num_genres, dtype=np.float32)
    for mid in watched_movies:
        if mid in movie_genre_map: pref_vec += movie_genre_map[mid]
    s = pref_vec.sum()
    if s > 0: pref_vec /= s
    return np.concatenate([user_feat, pref_vec]).astype(np.float32)

trajectories_all = []
for uid, g in pilot_ratings.groupby('user_id'):
    if len(g) >= 5:
        trajectories_all.append({'user_id': uid, 'movies': g['movie_id'].tolist(), 'ratings': g['rating'].tolist()})

candidate_movies = np.array(sorted(pilot_ratings['movie_id'].unique()))
num_actions = len(candidate_movies)


In [ ]:
def get_multiplier_demo(gender, age, occupation):
    m = 1.0
    if gender == 'F': m *= 1.36
    if age == 1: m *= 1.0
    elif age == 18: m *= 1.65
    elif age == 25: m *= 1.60
    elif age == 35: m *= 1.55
    elif age == 45: m *= 1.22
    elif age == 50: m *= 1.0
    elif age == 56: m *= 0.92
    if occupation in (1, 4, 6, 10, 11, 15): m *= 1.22
    return m

def get_train_forget_split(split_mode, cfg):
    su = np.array(sorted(ratings_df['user_id'].unique().tolist()))
    
    if split_mode == 'normal':
        set_seed(SEED)
        np.random.shuffle(su)
        split_amt = int(np.round(1 / 100 * len(su)))
        return set(su[:split_amt]), set(su[split_amt:])
        
    elif split_mode == 'demography':
        users_meta = users_df[users_df['user_id'].isin(su)].set_index('user_id')
        uid_mult = [(uid, get_multiplier_demo(users_meta.loc[uid]['gender'], users_meta.loc[uid]['age'], users_meta.loc[uid]['occupation'])) for uid in su]
        mean_mult = float(np.mean([m for _, m in uid_mult]))
        base_prob = (1 / 100) / mean_mult
        
        rng = np.random.default_rng(make_seed('forget_split'))
        f_users, r_users = [], []
        for uid, mult in uid_mult:
            p = min(base_prob * mult, 0.30)
            if rng.random() < p: f_users.append(uid)
            else: r_users.append(uid)
        return set(f_users), set(r_users)
        
    elif split_mode == 'ugp_setting':
        users_meta = users_df[users_df['user_id'].isin(su)].set_index('user_id')
        subset = users_meta[users_meta[cfg['setting_type']] == cfg['setting_value_raw']].copy().sort_index()
        f_users = subset.index.to_numpy(dtype=int)[:60].tolist()
        r_users = [u for u in su if u not in f_users]
        return set(f_users), set(r_users)


In [ ]:
def unlearning_finetune_ye_multi(policy_net, forget_buffer, retain_buffer, candidate_movies, num_iters, batch_size, lambda_retain, lr):
    device = next(policy_net.parameters()).device
    old_net = copy.deepcopy(policy_net).eval().to(device)
    optimizer = torch.optim.Adam(policy_net.parameters(), lr=lr)

    for it in range(num_iters):
        if len(forget_buffer) < batch_size or len(retain_buffer) < batch_size: break
        s_f, _, _, _, _ = forget_buffer.sample(batch_size, device)
        s_r, _, _, _, _ = retain_buffer.sample(batch_size, device)

        loss_forget = policy_net(s_f).abs().max(dim=1).values.mean()
        with torch.no_grad(): q_r_old = old_net(s_r)
        loss_retain = (policy_net(s_r) - q_r_old).abs().max(dim=1).values.mean()
        loss = loss_forget + lambda_retain * loss_retain

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def unlearning_finetune_new_true_inf(policy_net, forget_buffer, retain_buffer, candidate_movies, num_iters, batch_size, lambda_retain, lr):
    device = next(policy_net.parameters()).device
    old_net = copy.deepcopy(policy_net).eval().to(device)
    optimizer = torch.optim.Adam(policy_net.parameters(), lr=lr)

    for it in range(num_iters):
        if len(forget_buffer) < batch_size or len(retain_buffer) < batch_size: break
        s_f, _, _, _, _ = forget_buffer.sample(batch_size, device)
        s_r, _, _, _, _ = retain_buffer.sample(batch_size, device)

        loss_forget = policy_net(s_f).abs().max(dim=1).values.pow(2).mean()
        with torch.no_grad(): q_r_old = old_net(s_r)
        loss_retain = (policy_net(s_r) - q_r_old).abs().max(dim=1).values.pow(2).mean()
        loss = loss_forget + lambda_retain * loss_retain

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def unlearning_gradient_ascent(env, policy_net, num_iters, batch_size, lr, gamma, max_steps_per_ep):
    device = next(policy_net.parameters()).device
    optimizer = torch.optim.Adam(policy_net.parameters(), lr=lr)

    ep = 0
    while ep < num_iters:
        batch_log_probs, batch_returns = [], []
        for _ in range(batch_size):
            if ep >= num_iters: break
            state = env.reset()
            lps, rews = [], []
            for _ in range(max_steps_per_ep):
                st = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                probs = F.softmax(policy_net(st), dim=-1).squeeze(0)
                m = Categorical(probs)
                a = m.sample()
                next_state, rew, done = env.step(a.item())
                lps.append(m.log_prob(a))
                rews.append(rew)
                if done: break
                state = next_state

            G, rets = 0.0, []
            for r in reversed(rews):
                G = r + gamma * G
                rets.insert(0, G)
            rets = torch.tensor(rets, dtype=torch.float32).to(device)
            if len(rets) > 1:
                rets = (rets - rets.mean()) / (rets.std() + 1e-10)

            batch_log_probs.append(torch.stack(lps))
            batch_returns.append(rets)
            ep += 1

        if not batch_log_probs: break
        loss_forget = sum((lp * r).sum() for lp, r in zip(batch_log_probs, batch_returns))
        optimizer.zero_grad()
        loss_forget.backward()
        optimizer.step()


In [ ]:
all_metrics = []

for scenario, scenario_cfg in HARDCODED_BEST_CONFIGS.items():
    split_mode = scenario_cfg['split_mode']
    source_csv = scenario_cfg.get('source_csv', '')
    for method, cfg in scenario_cfg['methods'].items():
        print(f'\n--- Running {scenario} | {method} ---')
        
        # 1. Obtain data split
        f_users, r_users = get_train_forget_split(split_mode, cfg)
        f_trajs = [t for t in trajectories_all if t['user_id'] in f_users]
        r_trajs = [t for t in trajectories_all if t['user_id'] in r_users]
        print(f'Forget users: {len(f_trajs)}, Retain users: {len(r_trajs)}')
        
        # 2. Train Base Model exactly mimicking original process
        t_lr, gamma, hidden_dim, train_bs = cfg['train_lr'], cfg['gamma'], cfg['hidden_dim'], cfg['train_batch']
        
        # Set exact seed phase1
        set_seed(make_seed(t_lr, gamma, hidden_dim, train_bs, 'phase1'))
        
        net = PolicyNet(state_dim, num_actions, hidden_dim=hidden_dim).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=t_lr)
        
        env_tr = MovieLensEnv(trajectories_all, build_state_fn, candidate_movies)
        print('Training base model...')
        
        train_time_start = time.time()
        train_policy_gradient_batched(env_tr, net, opt, num_episodes=10000, gamma=gamma, batch_size=train_bs)
        train_time_s = time.time() - train_time_start
        
        base_model_path = os.path.join(MODELS_DIR, f'base_{scenario}_{method}.pt')
        torch.save(net.state_dict(), base_model_path)
        
        # Eval Base Model
        set_seed(make_seed(t_lr, gamma, hidden_dim, train_bs, 'eval'))
        net.eval()
        base_evals = eval_all_ks(net, r_trajs, f_trajs, trajectories_all)
        print(f'Base model K=10 Combined NDCG: {base_evals[10][5]:.4f}')
        
        # 3. Unlearning Loop N Times
        for run_idx in range(N_RUNS):
            print(f'-> Unlearning Run {run_idx + 1}/{N_RUNS}')
            
            # Setup specific seed
            var_seed = make_seed(t_lr, gamma, hidden_dim, train_bs, method, run_idx, 'unlearn')
            set_seed(var_seed)
            
            # Recreate buffers
            f_buf = ReplayBuffer(capacity=100000)
            env_f = MovieLensEnv(f_trajs, build_state_fn, candidate_movies)
            collect_random_experience_forget(env_f, num_steps=100000, buffer=f_buf)
            
            r_buf = ReplayBuffer(capacity=150000)
            env_r = MovieLensEnv(r_trajs, build_state_fn, candidate_movies)
            collect_policy_experience(env_r, net, num_steps=100000, buffer=r_buf)
            
            # Clone model for unlearning
            net_copy = copy.deepcopy(net)
            
            # Unlearn
            u_lr, u_iters, lam = cfg['unlearn_lr'], cfg['unlearn_iters'], cfg['lambda_retain']
            unlearn_time_start = time.time()
            if method == 'Ye_multi':
                unlearning_finetune_ye_multi(net_copy, f_buf, r_buf, candidate_movies, u_iters, UNLEARN_BATCH, lam, u_lr)
            elif method == 'New_True_inf':
                unlearning_finetune_new_true_inf(net_copy, f_buf, r_buf, candidate_movies, u_iters, UNLEARN_BATCH, lam, u_lr)
            elif method == 'Gradient_Ascent':
                unlearning_gradient_ascent(env_f, net_copy, u_iters, UNLEARN_BATCH, u_lr, gamma, MAX_STEPS)
            unlearn_time_s = time.time() - unlearn_time_start
            
            unlearned_model_path = os.path.join(MODELS_DIR, f'unlearn_{scenario}_{method}_run{run_idx}.pt')
            torch.save(net_copy.state_dict(), unlearned_model_path)
            
            # Evaluate after
            after_evals = eval_all_ks(net_copy, r_trajs, f_trajs, trajectories_all)
            
            # Log results for each K
            for K in KS:
                bh_r, bn_r, bh_f, bn_f, bh_c, bn_c = base_evals[K]
                ah_r, an_r, ah_f, an_f, ah_c, an_c = after_evals[K]
                
                all_metrics.append({
                    'scenario': scenario,
                    'method': method,
                    'run_idx': run_idx,
                    'K': K,
                    'train_lr': t_lr, 'gamma': gamma, 'hidden_dim': hidden_dim, 'train_batch': train_bs,
                    'unlearn_lr': u_lr, 'unlearn_iters': u_iters, 'lambda_retain': lam,
                    'setting_type': cfg.get('setting_type', ''),
                    'setting_value_raw': cfg.get('setting_value_raw', ''),
                    'setting_label': cfg.get('setting_label', ''),
                    'source_csv': source_csv,
                    'reason': cfg.get('reason', ''),
                    'reference_retain_drop_pp': cfg.get('retain_drop_pp', ''),
                    'reference_forget_drop_pp': cfg.get('forget_drop_pp', ''),
                    'base_retain_Hit': bh_r, 'base_retain_NDCG': bn_r,
                    'base_forget_Hit': bh_f, 'base_forget_NDCG': bn_f,
                    'base_combined_Hit': bh_c, 'base_combined_NDCG': bn_c,
                    'after_retain_Hit': ah_r, 'after_retain_NDCG': an_r,
                    'after_forget_Hit': ah_f, 'after_forget_NDCG': an_f,
                    'after_combined_Hit': ah_c, 'after_combined_NDCG': an_c,
                    'retain_drop_hit_pp': (bh_r - ah_r)*100,
                    'forget_drop_hit_pp': (bh_f - ah_f)*100,
                    'train_time_s': train_time_s,
                    'unlearn_time_s': unlearn_time_s,
                    'base_model_path': base_model_path,
                    'unlearned_model_path': unlearned_model_path
                })
            
            # Save intermediate metrics to ensure it updates during loop
            pd.DataFrame(all_metrics).to_csv(METRICS_CSV, index=False)

print('All runs completed successfully!')